In [33]:
!pip install flask flask-cors pyngrok nest-asyncio

from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import nest_asyncio
import pandas as pd
import joblib


nest_asyncio.apply()
app = Flask(__name__)
CORS(app)


model = joblib.load('/content/traffic_model.pkl')
label_encoder = joblib.load('/content/label_encoder.pkl')

@app.route('/')
def home():
    return jsonify({
        "message": "AI Traffic Congestion Prediction API Running"
    })

@app.route('/predict', methods=['POST'])
def predict():

    data = request.json

    input_df = pd.DataFrame([data])

    prediction = model.predict(input_df)

    congestion = label_encoder.inverse_transform(prediction)

    return jsonify({
        "predicted_congestion": congestion[0]
    })


In [34]:
import threading
import time

public_url = ngrok.connect(5000)
print("Public URL:", public_url)

def run_flask_app():
    app.run(host='0.0.0.0', port=5000)

flask_thread = threading.Thread(target=run_flask_app)
flask_thread.daemon = True
flask_thread.start()

time.sleep(2)

Public URL: NgrokTunnel: "https://euphemism-tropical-compound.ngrok-free.dev" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


In [35]:
import requests

url = "https://euphemism-tropical-compound.ngrok-free.dev/predict"

data = {
    "Location": "Connaught Place",
    "Latitude": 28.6315,
    "Longitude": 77.2167,
    "Traffic Volume": 700,
    "Avg Speed (km/h)": 12,
    "Weather": "Rain",
    "Rain(mm)": 15,
    "Accident": "Yes",
    "Event": "No",
    "Public Transport Density": 60,
    "Hour": 18,
    "DayOfWeek": 5,
    "Area": "Urban" # Added missing 'Area' column
}

response = requests.post(url, json=data)

print(response.json())

INFO:werkzeug:127.0.0.1 - - [14/May/2026 12:22:55] "POST /predict HTTP/1.1" 200 -


{'predicted_congestion': 'Very High'}
